# RL Systems 07 — rollout→train 的系统骨架

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Papa-Panda/field-notes/blob/main/pytorch/rl-systems/07-rl-systems.ipynb)

- **日期**：2026-10-02
- **定位**：RL track 的 RL systems coding 轮：不写新 loss，把"生成—打分—训练"三段搭成能跑的系统，接口契约和同步/异步的取舍讲清楚。
- **环境**：纯 PyTorch，CPU 单进程模拟分布式时序（看的是机制，不是吞吐）。

配套笔记：[07-rl-systems.md](07-rl-systems.md)


## 0. 因果链定位

前几篇的 loss 都默认"样本已经躺在手边"。在线 RL 的样本是 policy 自己生成的，于是多出 SFT 没有的系统问题：

- **生成（rollout）**：自回归解码，瓶颈在显存带宽，要靠推理引擎喂饱；
- **训练（learner）**：大矩阵乘，瓶颈在算力。

两个负载画像完全不同，绑在一起轮流跑就是互相等待。这一轮考的就是：怎么拆、接口存什么、同步还是异步、旧样本怎么修正。

玩具设定沿用 Research 03 的 bandit（6 个动作选目标动作 3），把结构换成 rollout/store/learner 三段。


## 1. 接口契约：Trajectory 里必须存什么

prompt、completion token、**采样时的 logp**、reward、**策略版本号** —— 后两样是异步修正和 debug 的命根子：没 logp 算不了重要性 ratio，没版本号不知道样本旧到什么程度。


In [ ]:
from dataclasses import dataclass, field
import torch

@dataclass
class Trajectory:
    """一条采样样本的契约：异步 RL 里 learner 能信任的信息全在这五个字段里。"""
    prompt_id: int
    action: int                 # toy 版 completion：一个动作
    old_logp: float             # 采样时行为策略给出的 logp（off-policy 修正要用）
    reward: float
    policy_version: int         # 采样时策略的版本号（新鲜度过滤要用）

@dataclass
class TrajectoryStore:
    """最简样本池：add 攒样本、drain 一次性取空，模拟 rollout → learner 的交接。"""
    buf: list = field(default_factory=list)
    def add(self, trajs): self.buf.extend(trajs)
    def drain(self):
        out, self.buf = self.buf, []
        return out


## 2. 同步版：rollout 一批 → 训练 → 再 rollout

样本永远 on-policy（版本差 0），结构简单。代价是 learner 训练时 rollout 在等、rollout 时 learner 在等 —— 先把这个基线跑通。


In [ ]:
def grpo_advantages(rewards):  # 与 Research 03 同款：组内标准化
    """rewards: (P, G) —— P 个 prompt、每个 prompt 采 G 个样本的 reward。
    返回: (P, G) 组内标准化 advantage（同组样本互为基线）。"""
    m = rewards.mean(dim=1, keepdim=True)
    s = rewards.std(dim=1, keepdim=True)
    return (rewards - m) / (s + 1e-8)

torch.manual_seed(0)
K, TARGET, G = 6, 3, 16
logits = torch.zeros(K, requires_grad=True)
opt = torch.optim.Adam([logits], lr=0.1)
version = 0
store = TrajectoryStore()

for round_ in range(6):
    # ---- rollout：用当前策略采样 G 条，写进 store ----
    with torch.no_grad():
        lp = torch.log_softmax(logits, -1)
        actions = torch.multinomial(lp.exp(), G, replacement=True)
        for a in actions.tolist():
            store.add([Trajectory(0, a, lp[a].item(), 1.0 if a == TARGET else 0.0, version)])
    # ---- learner：消费 store，GRPO 更新，版本 +1 ----
    trajs = store.drain()
    adv = grpo_advantages(torch.tensor([t.reward for t in trajs]).view(1, -1)).view(-1)
    lp = torch.log_softmax(logits, -1)
    loss = -(torch.stack([lp[t.action] for t in trajs]) * adv).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    version += 1
    print(f'round {round_}: P(目标动作) = {lp.exp()[TARGET].item():.3f}  （样本版本差恒为 0 = on-policy）')


## 3. 异步版：样本会旧，旧到什么程度要量化

改成：learner 每训一步，rollout worker 还在用 $ k $ 步前的权重生成。样本带着旧版本号进来，learner 有两个动作：

1. **算滞后** $ \text{lag} = v_{learner} - v_{sample} $ ，太旧的直接丢；
2. **重要性修正**：真实 ratio $ \rho = \exp(\log\pi_{new} - \log\pi_{old}) $ ，截断在 $[1-\epsilon, 1+\epsilon] $ 内，防旧样本把更新带偏。


In [ ]:
torch.manual_seed(0)
logits = torch.zeros(K, requires_grad=True)
opt = torch.optim.Adam([logits], lr=0.1)
snapshot = logits.detach().clone()   # rollout 用的权重快照（旧策略）
snapshot_version = 0                 # 快照对应的版本号
version = 0                          # learner 当前版本
lag_hist, kept = [], 0

for step in range(60):
    # rollout 用 snapshot 生成，trajectory 记快照的版本号
    with torch.no_grad():
        lp_old = torch.log_softmax(snapshot, -1)
        actions = torch.multinomial(lp_old.exp(), G, replacement=True)
    lp_new_all = torch.log_softmax(logits, -1)
    rewards = torch.tensor([1.0 if a == TARGET else 0.0 for a in actions.tolist()]).view(1, -1)
    adv = grpo_advantages(rewards).view(-1)

    lag = version - snapshot_version          # 这批样本旧了多少个版本
    if lag <= 5:                              # 新鲜度过滤：太旧整批丢弃
        losses = []
        for a, old_lp_i, adv_i in zip(actions.tolist(), lp_old[actions].tolist(), adv.tolist()):
            ratio = torch.exp(lp_new_all[a] - old_lp_i)
            ratio = torch.clamp(ratio, 0.8, 1.2)     # 重要性截断
            losses.append(-ratio * adv_i)
        loss = torch.stack(losses).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        kept += G
    lag_hist.extend([lag] * G)
    version += 1
    if version % 20 == 0:                     # 权重推送：每 20 步给 rollout 换新权重
        snapshot = logits.detach().clone()
        snapshot_version = version

import collections
print('样本滞后分布（lag: 条数）:', dict(sorted(collections.Counter(lag_hist).items())))
print(f'保留样本比例: {kept / len(lag_hist):.2f} | 最终 P(目标动作) = {torch.log_softmax(logits, -1).exp()[TARGET].item():.3f}')


## 4. Debug 演练：三个可跑的事故现场

事故清单先别背，下面三个事故都跑一遍，症状看清楚了清单自然记得住。修复都在各自正下方。

In [ ]:
# BUG 1：盲训旧样本（关掉新鲜度过滤）—— 先看它在这个 toy 里到底坏在哪
def run_async(use_filter):
    torch.manual_seed(0)
    logits = torch.zeros(K, requires_grad=True)
    opt = torch.optim.Adam([logits], lr=0.1)
    snapshot = logits.detach().clone(); snapshot_version = 0; version = 0
    used, stale = 0, 0
    for step in range(60):
        with torch.no_grad():
            lp_old = torch.log_softmax(snapshot, -1)
            actions = torch.multinomial(lp_old.exp(), G, replacement=True)
        lp_new_all = torch.log_softmax(logits, -1)
        rewards = torch.tensor([1.0 if a == TARGET else 0.0 for a in actions.tolist()]).view(1, -1)
        adv = grpo_advantages(rewards).view(-1)
        lag = version - snapshot_version
        if (not use_filter) or lag <= 5:                  # ← use_filter=False 时这行形同虚设
            losses = []
            for a, o_i, adv_i in zip(actions.tolist(), lp_old[actions].tolist(), adv.tolist()):
                ratio = torch.clamp(torch.exp(lp_new_all[a] - o_i), 0.8, 1.2)
                losses.append(-ratio * adv_i)
            loss = torch.stack(losses).mean()
            opt.zero_grad(); loss.backward(); opt.step()
            used += G
            if lag > 5: stale += G
        version += 1
        if version % 20 == 0:
            snapshot = logits.detach().clone(); snapshot_version = version
    return torch.log_softmax(logits, -1).exp()[TARGET].item(), used, stale

p_all, u_all, s_all = run_async(False)
p_filt, u_filt, _ = run_async(True)
print(f'BUG 1 观测: 不过滤 P(目标) = {p_all:.3f}（用了 {u_all} 条，其中 lag>5 的旧样本 {s_all} 条）')
print(f'          开过滤 P(目标) = {p_filt:.3f}（只用了 {u_filt} 条）—— toy 里盲训没翻车，甚至略快')
print('          真正的症状不在这条曲线：在于拿掉版本号之后，lag 分布根本算不出来，事故时无从定位')

**修复 1**：样本进 learner 前先打版本戳、按 lag 过滤：`policy_version` 写进 Trajectory（第 1 节契约），`lag > max_lag` 的整批丢弃（第 3 节用的 5）。

**为什么有效（以及它的真实代价）**：先诚实看上面的数——这个 toy 里盲训没翻车，因为 reward 与策略无关、ratio 截断又在兜底，过滤反而丢了 70% 样本。新鲜度过滤真正防的是漂移变大的场面：样本越旧，采样分布离当前策略越远，个别 ratio 会冲到几倍几十倍（BUG 3 的现场），那时没有过滤就是拿训练稳定性赌运气。还有一层更基础的：**没有版本戳，lag 分布根本算不出来**——线上出事时你连"样本有多旧"都回答不了，过滤阈值更无从调。所以顺序是：先打戳（可观测），再观测分布，最后才定阈值。

In [ ]:
# BUG 2：old_logp 不在采样时记录，让 learner 用自己的引擎重算（两边数值差 δ）
torch.manual_seed(0)
logits_now = torch.tensor([0.3, -0.2, 0.1, 0.8, -0.5, 0.0])
lp_learner = torch.log_softmax(logits_now, -1)          # learner 引擎算的 logp
delta = 0.1                                             # rollout 引擎（bf16/不同 kernel）与 learner 的系统性差
lp_rollout = lp_learner - delta                         # 采样当时真实的 logp（每个 token 都差这么多）
actions = torch.multinomial(lp_rollout.exp(), 16, replacement=True)
ratio_recomputed = torch.exp(lp_learner[actions] - lp_learner[actions])       # 用重算值当 old_logp
ratio_true = torch.exp(lp_learner[actions] - lp_rollout[actions])              # 用采样时记录的 old_logp
print('BUG 2 症状: 策略明明没变，重算版 ratio 恒 =', ratio_recomputed.mean().item(), '（自检假通过），真实 ratio =', round(ratio_true.mean().item(), 3), '（≠1：每个样本被系统性放大 e^0.1 倍）')

**修复 2**：old_logp 在**采样那一刻**由 rollout 引擎算好、写进 Trajectory 的 `old_logp` 字段（第 1 节契约），learner 直接用，不许重算；上线前再做一次一致性对拍：同一批 token 两边各算一遍 logp，差应 ≈ 0，超过阈值先停机查引擎差异。

**为什么有效**：importance ratio 的定义是 π_new(a)/π_behavior(a)，分母必须是"采样时那个策略"给出的概率。用 learner 重算，算出来的是 π_new/π_new ≡ 1 —— off-policy 修正被静默关掉，日志上还一切正常（上面 ratio 恒 1 的假通过就是这个指纹）。引擎间的数值差（精度、kernel、tokenizer 版本）还会再叠加一层系统性偏移，δ 只要 0.1，每条样本的权重就集体偏 10%。

In [ ]:
# BUG 3：importance ratio 不截断，旧样本权重爆炸
torch.manual_seed(0)
logits_new = torch.tensor([0., 0., 0., 2.5, 0., 0.])   # 已训了多步的当前策略
logits_old = torch.zeros(6)                             # rollout 还在用的旧快照
lp_n = torch.log_softmax(logits_new, -1); lp_o = torch.log_softmax(logits_old, -1)
acts = torch.multinomial(lp_o.exp(), 64, replacement=True)
ratio = torch.exp(lp_n[acts] - lp_o[acts])
print('BUG 3 症状: ratio 最大 =', round(ratio.max().item(), 2), '，中位数只有', round(ratio.median().item(), 2), '—— 个别旧样本的话语权是普通样本的近 10 倍，一条就能带偏整个 batch 的梯度')
print('         截断到 [0.8, 1.2] 后最大 =', torch.clamp(ratio, 0.8, 1.2).max().item())

**修复 3**：ratio 进 loss 前截断：`torch.clamp(ratio, 0.8, 1.2)`（第 3 节异步版就在做这件事，本质是把 PPO 的 clip 搬进重要性采样）。

**为什么有效**：样本越旧，采样分布和当前策略差越远，个别 ratio 会冲到 5 倍、10 倍——一条旧样本的梯度就能把整个 batch 带跑，这正是异步 RL 训练曲线突然 spikes 的头号来源。截断等于给每条样本的话语权封顶，代价是引入一点偏差，换来方差可控。两把锁要一起用：版本过滤管"太旧的不要"，ratio 截断管"留下的也别喊太大声"。

## 5. 事故清单（速查，修复看第 4 节）

可跑的三条都在第 4 节：盲训旧样本（BUG 1）、old_logp 重算致 ratio 假通过（BUG 2）、ratio 不截断权重爆炸（BUG 3）。剩下两条没有现场版，记住处置即可：

- **权重推送竞态** —— 生成中途换权重，同一条 trajectory 前半旧策略后半新策略。**修复**：按版本切分生成批次，换权重只在批次边界发生。
- **reward 拖死流水线** —— reward 是外部服务，一批卡住全体等。**修复**：慢调用必须有超时 + 降级（如先用旧 reward 模型顶着）。

## 小结

- RL 系统的主线问题是负载错配：生成吃带宽、训练吃算力，拆开部署是默认答案。
- Trajectory 契约五件套：prompt、token、采样 logp、reward、版本号 —— 缺一个，异步就盲。
- 异步的两把锁：版本滞后过滤 + 重要性截断；同步的两笔账：利用率气泡。
- 权重推送决定滞后分布：它不是运维细节，是算法正确性的一部分。

**至此阶段 1–3 的 notebook 全部就位**（Research 01–04、Day 01 DDP、Parallel 05/06/08、RL Systems 07）。剩下的只有阶段 4 的缺口：system design、理论、behavioral —— 那些不写代码，单独准备。